In [2]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# CONFIGURAÇÕES
# ============================================================

PASTA_CSV = r"dados\raw\cliente\csv"

PASTA_PARQUET = r"dados\staging\parquet"

os.makedirs(
    PASTA_PARQUET,
    exist_ok=True
)


# ============================================================
# FUNÇÃO PARA SALVAR PARQUET
# ============================================================

def salvar_parquet_string(df, caminho_saida):

    # --------------------------------------------------------
    # Garante que valores nulos sejam vazios
    # --------------------------------------------------------

    df = df.fillna("")

    # --------------------------------------------------------
    # Converte todas as colunas para STRING
    # --------------------------------------------------------

    for coluna in df.columns:

        df[coluna] = (
            df[coluna]
            .astype(str)
            .str.strip()
        )

    # --------------------------------------------------------
    # Normaliza nomes das colunas
    # --------------------------------------------------------

    df.columns = [
        str(coluna).strip()
        for coluna in df.columns
    ]

    # --------------------------------------------------------
    # Remove colunas duplicadas
    # --------------------------------------------------------

    df = df.loc[
        :,
        ~df.columns.duplicated()
    ]

    # --------------------------------------------------------
    # Schema exclusivamente STRING
    # --------------------------------------------------------

    schema = pa.schema(
        [
            pa.field(
                coluna,
                pa.string()
            )
            for coluna in df.columns
        ]
    )

    # --------------------------------------------------------
    # Converte para PyArrow
    # --------------------------------------------------------

    tabela = pa.Table.from_pandas(
        df,
        schema=schema,
        preserve_index=False
    )

    # --------------------------------------------------------
    # Salva Parquet
    # --------------------------------------------------------

    pq.write_table(
        tabela,
        caminho_saida,
        compression="snappy"
    )

    return tabela


# ============================================================
# PROCESSAMENTO DOS CSV
# ============================================================

for arquivo in os.listdir(PASTA_CSV):

    # --------------------------------------------------------
    # Processa somente arquivos CSV
    # --------------------------------------------------------

    if not arquivo.lower().endswith(".csv"):
        continue

    caminho_arquivo = os.path.join(
        PASTA_CSV,
        arquivo
    )

    nome_base = os.path.splitext(
        arquivo
    )[0]

    print("\n" + "=" * 70)
    print(f"Processando: {arquivo}")
    print("=" * 70)

    try:

        # ====================================================
        # 1. IDENTIFICA O HEADER DA SEGUNDA TABELA
        # ====================================================

        with open(
            caminho_arquivo,
            "r",
            encoding="utf-8",
            errors="replace"
        ) as f:

            linhas = f.readlines()

        linha_header = None

        for i, linha in enumerate(linhas):

            linha_lower = linha.lower()

            if (
                "data" in linha_lower
                and "lat" in linha_lower
            ):

                linha_header = i
                break

        if linha_header is None:

            print(
                "⚠ Cabeçalho 'data' e 'lat' não encontrado."
            )

            continue

        print(
            f"Header da segunda tabela: linha {linha_header}"
        )


        # ====================================================
        # 2. EXTRAI A PRIMEIRA TABELA - METADADOS
        # ====================================================
        #
        # IMPORTANTE:
        # Não usamos skiprows=1.
        #
        # Dessa forma a PRIMEIRA LINHA também é capturada.
        #
        # ====================================================

        df_meta = pd.read_csv(
            caminho_arquivo,
            nrows=linha_header,
            header=None,
            encoding="utf-8",
            engine="python"
        )

        # ----------------------------------------------------
        # Remove colunas completamente vazias
        # ----------------------------------------------------

        df_meta = df_meta.dropna(
            axis=1,
            how="all"
        )

        # ----------------------------------------------------
        # Remove linhas completamente vazias
        # ----------------------------------------------------

        df_meta = df_meta.dropna(
            axis=0,
            how="all"
        )

        # ----------------------------------------------------
        # Garante que todas as colunas sejam string
        # ----------------------------------------------------

        df_meta = df_meta.fillna("")

        for coluna in df_meta.columns:

            df_meta[coluna] = (
                df_meta[coluna]
                .astype(str)
                .str.strip()
            )

        # ----------------------------------------------------
        # Mostra quantidade original
        # ----------------------------------------------------

        print(
            f"Primeira tabela capturada: "
            f"{len(df_meta)} linhas, "
            f"{len(df_meta.columns)} colunas"
        )

        # ====================================================
        # 3. TRANSPÕE A PRIMEIRA TABELA
        # ====================================================
        #
        # A primeira coluna contém os nomes dos campos.
        #
        # Exemplo:
        #
        # nome_arquivo | valor
        # relatorio    | xxx
        # data         | xxx
        #
        # Após transposição:
        #
        # nome_arquivo | relatorio | data
        #
        # ====================================================

        df_meta.columns = [
            f"coluna_{i}"
            for i in range(len(df_meta.columns))
        ]

        # ----------------------------------------------------
        # Usa a primeira coluna como nome dos campos
        # ----------------------------------------------------

        df_meta = (
            df_meta
            .set_index("coluna_0")
            .T
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Normaliza nomes das colunas
        # ----------------------------------------------------

        df_meta.columns = [
            str(col).strip()
            for col in df_meta.columns
        ]

        # ----------------------------------------------------
        # Remove colunas duplicadas
        # ----------------------------------------------------

        df_meta = df_meta.loc[
            :,
            ~df_meta.columns.duplicated()
        ]

        print(
            f"Primeira tabela após transposição: "
            f"{len(df_meta)} registros, "
            f"{len(df_meta.columns)} colunas"
        )


        # ====================================================
        # 4. SALVA PRIMEIRA TABELA
        # ====================================================

        caminho_meta = os.path.join(
            PASTA_PARQUET,
            f"{nome_base}_metadados.parquet"
        )

        tabela_meta = salvar_parquet_string(
            df_meta,
            caminho_meta
        )

        print(
            "✅ Primeira tabela salva:"
        )

        print(
            f"   {caminho_meta}"
        )


        # ====================================================
        # 5. EXTRAI SEGUNDA TABELA - DADOS
        # ====================================================

        df_dados = pd.read_csv(
            caminho_arquivo,
            skiprows=linha_header,
            encoding="utf-8",
            engine="python"
        )

        # ----------------------------------------------------
        # Remove colunas completamente vazias
        # ----------------------------------------------------

        df_dados = df_dados.dropna(
            axis=1,
            how="all"
        )

        # ----------------------------------------------------
        # Normaliza nomes das colunas
        # ----------------------------------------------------

        df_dados.columns = [
            str(col).strip()
            for col in df_dados.columns
        ]

        print(
            f"Segunda tabela: "
            f"{len(df_dados)} registros, "
            f"{len(df_dados.columns)} colunas"
        )


        # ====================================================
        # 6. SALVA SEGUNDA TABELA
        # ====================================================

        caminho_dados = os.path.join(
            PASTA_PARQUET,
            f"{nome_base}_dados.parquet"
        )

        tabela_dados = salvar_parquet_string(
            df_dados,
            caminho_dados
        )

        print(
            "✅ Segunda tabela salva:"
        )

        print(
            f"   {caminho_dados}"
        )


        # ====================================================
        # 7. MOSTRA SCHEMA METADADOS
        # ====================================================

        print("\nSchema METADADOS:")

        for campo in tabela_meta.schema:

            print(
                f"   {campo.name}: {campo.type}"
            )


        # ====================================================
        # 8. MOSTRA SCHEMA DADOS
        # ====================================================

        print("\nSchema DADOS:")

        for campo in tabela_dados.schema:

            print(
                f"   {campo.name}: {campo.type}"
            )


        # ====================================================
        # RESULTADO
        # ====================================================

        print("\n" + "-" * 70)

        print(
            "✅ PROCESSAMENTO CONCLUÍDO"
        )

        print(
            f"Metadados: "
            f"{nome_base}_metadados.parquet"
        )

        print(
            f"Dados: "
            f"{nome_base}_dados.parquet"
        )

        print("-" * 70)


    except Exception as e:

        print(
            f"\n❌ Erro ao processar {arquivo}:"
        )

        print(
            f"   {e}"
        )


# ============================================================
# FINALIZAÇÃO
# ============================================================

print("\n" + "=" * 70)

print(
    "PROCESSAMENTO FINALIZADO"
)

print("=" * 70)


Processando: CLIENTE1_relatorio1_dados.csv
Header da segunda tabela: linha 24
Primeira tabela capturada: 24 linhas, 2 colunas
Primeira tabela após transposição: 1 registros, 24 colunas
✅ Primeira tabela salva:
   dados\staging\parquet\CLIENTE1_relatorio1_dados_metadados.parquet
Segunda tabela: 3 registros, 6 colunas
✅ Segunda tabela salva:
   dados\staging\parquet\CLIENTE1_relatorio1_dados_dados.parquet

Schema METADADOS:
   nome_arquivo: string
   coluna2: string
   coluna3: string
   coluna4: string
   coluna5: string
   coluna6: string
   coluna7: string
   coluna8: string
   coluna9: string
   coluna10: string
   coluna11: string
   coluna12: string
   coluna13: string
   coluna14: string
   coluna15: string
   coluna16: string
   coluna17: string
   coluna18: string
   coluna19: string
   coluna21: string
   coluna22: string
   coluna23: string
   coluna24: string
   coluna25: string

Schema DADOS:
   data: string
   speed: string
   yaw: string
   lat: string
   long: string
   